In [9]:
"""
Descarga de clasificación del suelo para análisis de construcción
planta de biometano — Provincia de Huesca

Fuentes:
  1. ICEAragón WFS (SIUa) — clasificación urbanística

Output: clasificacion_suelo_huesca.gpkg

NOTA sobre clasificacion_idearagon:
  La capa SIUa:clasificacion_idearagon NO está disponible vía GetFeature JSON
  en el endpoint SIUa_WMS. Se prueban variantes de nombre y parámetros.
  Si todo falla, descarga manual en: https://idearagon.aragon.es/SIUa/descargas
"""

import geopandas as gpd
import requests
import io
from pathlib import Path
from shapely.ops import unary_union

# ─────────────────────────────────────────────────────────
# CONFIGURACIÓN
# ─────────────────────────────────────────────────────────
DELIM_DIR      = Path("../../data/raw/delimitations")
OUTPUT_DIR     = Path("../../data/raw/06_clasificacion_suelo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GEOJSON_HUESCA = DELIM_DIR / "Huesca_Delimitacion.geojson"
OUTPUT_GPKG    = OUTPUT_DIR / "clasificacion_suelo_huesca.gpkg"

CRS_UTM    = "EPSG:25830"
BBOX_WGS84 = (-0.934168, 41.347806, 0.771832, 42.921806)  # minx,miny,maxx,maxy
HEADERS    = {"User-Agent": "biometano-huesca-research/1.0 (academic)"}
WFS_SIUA = "https://icearagon.aragon.es/geoserver/wfs"


def cargar_huesca():
    huesca = gpd.read_file(GEOJSON_HUESCA).to_crs(CRS_UTM)
    return unary_union(huesca.geometry)


# ─────────────────────────────────────────────────────────
# 1. ICEAragón SIUa — CLASIFICACIÓN URBANÍSTICA
# ─────────────────────────────────────────────────────────
def descargar_clasificacion_urbanistica():
    print("Descargando clasificación urbanística (ICEAragón SIUa)...")

    WFS_URL = "https://icearagon.aragon.es/geoserver/wfs"

    # BBOX de Huesca en EPSG:25830 (UTM zona 30N)
    # minx, miny, maxx, maxy
    BBOX_UTM = (585000, 4580000, 790000, 4760000)

    params = {
        "SERVICE":      "WFS",
        "VERSION":      "2.0.0",
        "REQUEST":      "GetFeature",
        "TYPENAMES":    "SIUa:clasificacion_idearagon",
        "OUTPUTFORMAT": "application/json",
        "SRSNAME":      "EPSG:25830",
        "BBOX":         f"{BBOX_UTM[0]},{BBOX_UTM[1]},{BBOX_UTM[2]},{BBOX_UTM[3]},EPSG:25830",
    }
    try:
        r = requests.get(WFS_URL, params=params, timeout=300, headers=HEADERS)
        r.raise_for_status()

        if b"ExceptionReport" in r.content[:500]:
            print(f"  ExceptionReport:\n{r.text}")
            return None

        gdf = gpd.read_file(io.BytesIO(r.content))

        if gdf is None or len(gdf) == 0:
            print("  Respuesta vacía")
            return None

        if gdf.crs is None:
            gdf = gdf.set_crs("EPSG:25830")
        gdf = gdf.to_crs(CRS_UTM)
        print(f"  Total: {len(gdf)} polígonos")
        col_cls = next((c for c in gdf.columns
                        if any(k in c.lower() for k in
                               ["clasif", "tipo", "clase", "uso"])), None)
        if col_cls:
            print(f"  Valores '{col_cls}': {gdf[col_cls].value_counts().to_dict()}")
        return gdf

    except requests.HTTPError as e:
        print(f"  Error HTTP {e.response.status_code}:\n{e.response.text[:800]}")
    except Exception as e:
        print(f"  Error: {e}")

    print("  Sin datos. Descarga manual:")
    print("  → https://idearagon.aragon.es/SIUa/descargas  (seleccionar Huesca, formato GPKG)")
    return None


# ─────────────────────────────────────────────────────────
# GUARDAR
# ─────────────────────────────────────────────────────────
def guardar(gdf_urbanistico):
    print()
    capas = {
        "clasificacion_urbanistica": gdf_urbanistico,
    }
    for nombre, gdf in capas.items():
        if gdf is not None and len(gdf) > 0:
            gdf.to_file(OUTPUT_GPKG, driver="GPKG", layer=nombre)
            print(f"  '{nombre}': {len(gdf)} entidades → {OUTPUT_GPKG}")
        else:
            print(f"  '{nombre}': sin datos, no guardada")


# ─────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────
if __name__ == "__main__":

    print("\n" + "=" * 60)
    print("  CLASIFICACIÓN SUELO HUESCA — DESCARGA")
    print("=" * 60 + "\n")

    gdf_urbanistico = descargar_clasificacion_urbanistica()

    guardar(gdf_urbanistico)

    print("\nHecho. Capas en:", OUTPUT_GPKG)


  CLASIFICACIÓN SUELO HUESCA — DESCARGA

Descargando clasificación urbanística (ICEAragón SIUa)...
  Total: 33586 polígonos
  Valores 'clasesiose': {'NULL': 394, 'SU-C': 182, 'SNU-G': 100, 'SU': 88, 'SNU-C': 1, 'SNU-E': 1}

  'clasificacion_urbanistica': 33586 entidades → ..\..\data\raw\06_clasificacion_suelo\clasificacion_suelo_huesca.gpkg

Hecho. Capas en: ..\..\data\raw\06_clasificacion_suelo\clasificacion_suelo_huesca.gpkg


In [10]:
import numpy as np
import geopandas as gpd
import pandas as pd
import pydeck as pdk
from pathlib import Path

RAW_DIR = Path("../../data/raw/06_clasificacion_suelo")
MAP_DIR = Path("../../data/map/06_clasificacion_suelo")
RAW_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)

GPKG = RAW_DIR / "clasificacion_suelo_huesca.gpkg"
gdf = gpd.read_file(GPKG, layer="clasificacion_urbanistica")
gdf = gdf[gdf["cod_ine"].astype(str).str.startswith("22")].copy()  # ← aquí
gdf = gdf.to_crs("EPSG:4326")
gdf["geometry"] = gdf["geometry"].simplify(0.0003, preserve_topology=True)
gdf = gdf[gdf["clase"].notna()].copy()

CLASE_STYLE = {
    "SUC":   {"color": [52,  152, 219, 255]},   # azul vivo
    "SUNC":  {"color": [41,  128, 185, 255]},   # azul oscuro
    "SDUD":  {"color": [243, 156,  18, 255]},   # naranja vivo
    "SDUNC": {"color": [230, 126,  34, 255]},   # naranja oscuro
    "SENU":  {"color": [ 39, 174,  96, 140]},   # verde — más transparente
    "SENE":  {"color": [ 26, 188, 156, 140]},   # verde agua — más transparente
    "SEUND": {"color": [155,  89, 182, 255]},   # morado vivo
}
DEFAULT = {"color": [150, 150, 150, 150]}

gdf["fill_color"] = gdf["clase"].apply(
    lambda c: CLASE_STYLE.get(str(c).strip(), DEFAULT)["color"]
)

def geom_to_coords(geom):
    if geom is None or geom.is_empty:
        return None
    if geom.geom_type == "Polygon":
        return [list(geom.exterior.coords)]
    if geom.geom_type == "MultiPolygon":
        biggest = max(geom.geoms, key=lambda p: p.area)
        return [list(biggest.exterior.coords)]
    return None

gdf["coordinates"] = gdf["geometry"].apply(geom_to_coords)
gdf = gdf[gdf["coordinates"].notna()].copy()

df = gdf[["coordinates", "clase", "clase_s", "d_muni_ine", "uso_glob", "fill_color"]].copy()
df["clase"]      = df["clase"].fillna("—")
df["clase_s"]    = df["clase_s"].fillna("—")
df["d_muni_ine"] = df["d_muni_ine"].fillna("—")
df["uso_glob"]   = df["uso_glob"].fillna("—")

print(f"Polígonos: {len(df):,}")

layer = pdk.Layer(
    "PolygonLayer",
    data=df,
    get_polygon="coordinates",
    get_fill_color="fill_color",
    get_line_color=[60, 60, 60, 100],
    line_width_min_pixels=0.3,
    extruded=False,
    pickable=True,
    auto_highlight=True,
    wireframe=False,
)

view = pdk.ViewState(
    longitude=-0.08,
    latitude=42.10,
    zoom=8,
    min_zoom=6,
    max_zoom=14,
    pitch=0,
    bearing=0,
)

tooltip = {
    "html": """
        <b>Clase:</b> {clase}<br/>
        <b>Subtipo:</b> {clase_s}<br/>
        <b>Municipio:</b> {d_muni_ine}<br/>
        <b>Uso global:</b> {uso_glob}
    """,
    "style": {"backgroundColor": "#111", "color": "white",
              "fontSize": "12px", "padding": "6px"},
}

LEGEND_HTML = """
<div style="position:fixed;bottom:30px;left:20px;background:rgba(0,0,0,0.75);
            color:#fff;padding:12px 16px;border-radius:8px;font-size:12px;
            font-family:sans-serif;z-index:9999;line-height:2">
  <b>Clasificación del suelo — Huesca</b><br/>
  <span style="color:#3498db">██</span> SUC   — Urbano Consolidado<br/>
  <span style="color:#2980b9">██</span> SUNC  — Urbano No Consolidado<br/>
  <span style="color:#f39c12">██</span> SDUD  — Urbanizable Delimitado<br/>
  <span style="color:#e67e22">██</span> SDUNC — Urbanizable No Delimitado<br/>
  <span style="color:#27ae60">██</span> SENU  — No Urbanizable Genérico<br/>
  <span style="color:#1abc9c">██</span> SENE  — No Urbanizable Especial<br/>
  <span style="color:#9b59b6">██</span> SEUND — No Urbanizable Especial ND<br/>
</div>
"""

deck = pdk.Deck(
    layers=[layer],
    initial_view_state=view,
    tooltip=tooltip,
    map_style="https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json",
)

output_html = MAP_DIR / "mapa_clasificacion_huesca.html"
html = deck.to_html(as_string=True)
html = html.replace("</body>", LEGEND_HTML + "\n</body>")
with open(output_html, "w", encoding="utf-8") as f:
    f.write(html)

print(f"Abre: {output_html}")

Polígonos: 17,393
Abre: ..\..\data\map\06_clasificacion_suelo\mapa_clasificacion_huesca.html


In [33]:
import numpy as np
import requests
import geopandas as gpd
import pandas as pd
import pydeck as pdk
from pathlib import Path
import unicodedata
import re

RAW_DIR = Path("../../data/raw/06_clasificacion_suelo")
MAP_DIR = Path("../../data/map/06_clasificacion_suelo")
RAW_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)

# --- 1. Descargar padrón INE ---
print("Descargando padrón INE...")
url = "https://servicios.ine.es/wstempus/js/ES/DATOS_TABLA/2875?nult=1"
r = requests.get(url, timeout=60)
data = r.json()

registros = []
for item in data:
    nombre = item.get("Nombre", "")
    valor  = item.get("Data", [{}])[0].get("Valor", None)
    registros.append({"nombre": nombre, "poblacion": valor})

raw_df = pd.DataFrame(registros)

# La fila de TOTAL PROVINCIA tiene el mismo texto "Huesca." que el MUNICIPIO de Huesca
# capital, pero la provincia aparece siempre como la primera fila "Huesca. ..." de toda
# la tabla. La identificamos por posición (índice mínimo), no por texto, para no
# confundirla con el municipio homónimo.
es_fila_huesca_texto = raw_df["nombre"].str.strip().str.match(r"^Huesca\.\s")
idx_provincia = raw_df[es_fila_huesca_texto].index.min()

padron = raw_df.drop(index=idx_provincia).copy()

# Solo filas "Total" (sexo combinado) + "Total habitantes" (todas las edades)
padron = padron[
    padron["nombre"].str.contains(r"\. Total\. Total habitantes", regex=True)
].copy()

def quitar_tildes(s):
    """Quita diacríticos para comparación robusta (Á->A, Í->I, É->E, etc.)"""
    return "".join(
        c for c in unicodedata.normalize("NFKD", s)
        if not unicodedata.combining(c)
    )

def limpiar_nombre_ine(nombre_completo):
    """
    Convierte el nombre crudo del INE en un nombre normalizado y comparable
    con la capa geográfica, gestionando:
      - el artículo pospuesto con coma: "Fueva, La" -> "La Fueva"
      - el nombre bilingüe con barra: "Huesca/Uesca" -> "Huesca"
    """
    base = nombre_completo.split(".")[0].strip()
    base = base.split("/")[0].strip()

    m = re.match(r"^(.*),\s*(La|El|Los|Las)$", base, flags=re.IGNORECASE)
    if m:
        cuerpo, articulo = m.groups()
        base = f"{articulo} {cuerpo}"

    return base

padron["municipio_raw"] = padron["nombre"].apply(limpiar_nombre_ine)

def normalizar_texto(s):
    if pd.isna(s):
        return s
    s = unicodedata.normalize("NFKC", str(s))
    s = quitar_tildes(s)             # Á->A, Í->I, etc. — clave para AÍSA/AISA, COSTEÁN/COSTEAN
    s = s.replace("-", " ")          # unifica guión y espacio
    s = " ".join(s.split())          # colapsa espacios múltiples/invisibles y hace strip
    return s.upper()

padron["municipio"] = padron["municipio_raw"].apply(normalizar_texto)
padron["poblacion"] = pd.to_numeric(padron["poblacion"], errors="coerce").fillna(0).astype(int)
padron = padron[["municipio", "poblacion"]]

dup = padron["municipio"].duplicated().sum()
if dup:
    print(f"⚠ {dup} municipios con más de una fila en el padrón tras filtrar — revisa el filtro")
padron = padron.drop_duplicates(subset="municipio", keep="first")

print(f"Municipios con población: {len(padron)}")

# --- 2. Buffer por población según normativa Aragón ---
def buffer_por_poblacion(pop):
    if pop < 3000: return 1000
    else:          return 2000

GPKG = RAW_DIR / "clasificacion_suelo_huesca.gpkg"
gdf_raw = gpd.read_file(GPKG, layer="clasificacion_urbanistica")
gdf_raw = gdf_raw[gdf_raw["cod_ine"].astype(str).str.startswith("22")]
nucleos = gdf_raw[gdf_raw["clase"].isin(["SUC", "SUNC"])].copy()

nucleos["d_muni_ine"] = nucleos["d_muni_ine"].apply(normalizar_texto)  # ahora también quita tildes
nucleos["cod_ine"] = nucleos["cod_ine"].astype(str).str.strip()

nucleos_diss = nucleos.dissolve(by="cod_ine", aggfunc={"d_muni_ine": "first"}).reset_index()
nucleos_diss["municipio_key"] = nucleos_diss["d_muni_ine"].str.strip().str.upper()

dup_cod = nucleos_diss["cod_ine"].duplicated().sum()
if dup_cod:
    print(f"⚠ {dup_cod} cod_ine duplicados tras el dissolve — revisar")
else:
    print("✓ Dissolve por cod_ine correcto: cada municipio es una única geometría")

nucleos_diss = nucleos_diss.merge(padron, left_on="municipio_key",
                                   right_on="municipio", how="left")
nucleos_diss["poblacion"] = nucleos_diss["poblacion"].fillna(0).astype(int)
nucleos_diss["buffer_m"]  = nucleos_diss["poblacion"].apply(buffer_por_poblacion)

print(f"Municipios sin match padrón: {(nucleos_diss['poblacion']==0).sum()}")
print(nucleos_diss[["d_muni_ine", "poblacion", "buffer_m"]].sort_values("poblacion", ascending=False).head(10))

sin_match = nucleos_diss[nucleos_diss["poblacion"] == 0]
if len(sin_match) > 0:
    print(f"\n⚠ {len(sin_match)} municipios sin match:")
    print(sin_match["d_muni_ine"].tolist())

nucleos_diss["geometry"] = nucleos_diss.apply(
    lambda row: row.geometry.buffer(row["buffer_m"]), axis=1
)

BUFFER_GPKG = RAW_DIR / "buffer_nucleos_urbanos.gpkg"
nucleos_diss.to_file(BUFFER_GPKG, driver="GPKG")
print(f"Guardado: {BUFFER_GPKG}")

Descargando padrón INE...
Municipios con población: 202
✓ Dissolve por cod_ine correcto: cada municipio es una única geometría
Municipios sin match padrón: 0
             d_muni_ine  poblacion  buffer_m
91               HUESCA      55454      2000
109              MONZON      18525      2000
37            BARBASTRO      17807      2000
82                FRAGA      15576      2000
95                 JACA      14024      2000
49              BINEFAR      10235      2000
135          SABINANIGO       9695      2000
147            SARINENA       4113      2000
155  TAMARITE DE LITERA       3464      2000
87                GRAUS       3400      2000
Guardado: ..\..\data\raw\06_clasificacion_suelo\buffer_nucleos_urbanos.gpkg


In [36]:
for nombre in ["PERALTA DE ALCOFEA", "SARINENA"]:
    row = nucleos_diss[nucleos_diss["d_muni_ine"] == nombre].iloc[0]
    geom = row.geometry
    print(f"\n{nombre} — {len(geom.geoms)} partes:")
    for i, part in enumerate(geom.geoms):
        print(f"  parte {i}: área = {part.area:,.0f} m²  (radio equivalente ≈ {(part.area/3.1416)**0.5:,.0f} m)")


PERALTA DE ALCOFEA — 3 partes:
  parte 0: área = 4,179,236 m²  (radio equivalente ≈ 1,153 m)
  parte 1: área = 4,429,015 m²  (radio equivalente ≈ 1,187 m)
  parte 2: área = 5,961,418 m²  (radio equivalente ≈ 1,378 m)

SARINENA — 5 partes:
  parte 0: área = 29,347,844 m²  (radio equivalente ≈ 3,056 m)
  parte 1: área = 49,125,592 m²  (radio equivalente ≈ 3,954 m)
  parte 2: área = 15,456,928 m²  (radio equivalente ≈ 2,218 m)
  parte 3: área = 17,732,921 m²  (radio equivalente ≈ 2,376 m)
  parte 4: área = 17,236,403 m²  (radio equivalente ≈ 2,342 m)


In [34]:
# --- Mapa SOLO de buffers de núcleos urbanos (con límite de provincia) ---
DELIM_DIR = Path("../../data/raw/delimitations")
provincia = gpd.read_file(DELIM_DIR / "Huesca_Delimitacion.geojson").to_crs("EPSG:4326")

def geom_to_coords_multi(geom):
    """Devuelve una lista de anillos exteriores (uno por cada polígono del multipolígono)."""
    if geom is None or geom.is_empty:
        return None
    if geom.geom_type == "Polygon":
        return [list(geom.exterior.coords)]
    if geom.geom_type == "MultiPolygon":
        return [list(p.exterior.coords) for p in geom.geoms]
    return None

nucleos_buffer = nucleos_diss.to_crs("EPSG:4326")
nucleos_buffer["geometry"] = nucleos_buffer["geometry"].simplify(0.001, preserve_topology=True)
nucleos_buffer["coordinates"] = nucleos_buffer["geometry"].apply(geom_to_coords_multi)
nucleos_buffer = nucleos_buffer[nucleos_buffer["coordinates"].notna()].copy()

# Explode: una fila por cada anillo de polígono (necesario para MultiPolygon en pydeck)
buffer_rows = []
for _, row in nucleos_buffer.iterrows():
    for ring in row["coordinates"]:
        buffer_rows.append({
            "coordinates": [ring],
            "d_muni_ine": row["d_muni_ine"],
            "poblacion": row["poblacion"],
            "buffer_m": row["buffer_m"],
        })

df_buffer = pd.DataFrame(buffer_rows)
print(f"Polígonos de buffer a dibujar: {len(df_buffer)}")

# Color según el buffer aplicado (1000 m vs 2000 m) — tonos más suaves
def color_por_buffer(bm):
    return [230, 126, 34, 130] if bm == 1000 else [192, 57, 43, 170]

df_buffer["fill_color"] = df_buffer["buffer_m"].apply(color_por_buffer)

layer_buffer = pdk.Layer(
    "PolygonLayer",
    data=df_buffer,
    get_polygon="coordinates",
    get_fill_color="fill_color",
    get_line_color=[255, 255, 255, 90],
    line_width_min_pixels=0.5,
    extruded=False,
    pickable=True,
    auto_highlight=True,
)

# --- Límite de la provincia ---
boundary_features = []
for geom in provincia.geometry:
    polys = [geom] if geom.geom_type == "Polygon" else list(geom.geoms)
    for p in polys:
        boundary_features.append({
            "type": "Feature",
            "geometry": {"type": "Polygon", "coordinates": [list(p.exterior.coords)]},
            "properties": {}
        })

geojson_boundary = {"type": "FeatureCollection", "features": boundary_features}

layer_boundary = pdk.Layer(
    "GeoJsonLayer",
    data=geojson_boundary,
    stroked=True,
    filled=False,
    get_line_color=[255, 255, 255, 200],
    get_line_width=800,
    line_width_min_pixels=2,
)

view = pdk.ViewState(
    longitude=-0.08,
    latitude=42.05,
    zoom=7.6,
    min_zoom=6,
    max_zoom=14,
    pitch=0,
    bearing=0,
)

tooltip = {
    "html": """
        <b>Municipio:</b> {d_muni_ine}<br/>
        <b>Población:</b> {poblacion}<br/>
        <b>Buffer aplicado:</b> {buffer_m} m
    """,
    "style": {"backgroundColor": "#111", "color": "white",
              "fontSize": "12px", "padding": "6px"},
}

LEGEND_HTML = """
<div style="position:fixed;bottom:30px;left:20px;background:rgba(0,0,0,0.75);
            color:#fff;padding:14px 18px;border-radius:8px;font-size:13px;
            font-family:sans-serif;z-index:9999;line-height:1.9">
  <div style="font-weight:bold;font-size:14px;margin-bottom:8px;">Buffers de núcleos urbanos — Huesca</div>
  <div><span style="display:inline-block;width:14px;height:14px;background:#e67e22;border-radius:3px;margin-right:8px;vertical-align:middle;"></span>Buffer 1.000 m (&lt; 3.000 hab)</div>
  <div><span style="display:inline-block;width:14px;height:14px;background:#c0392b;border-radius:3px;margin-right:8px;vertical-align:middle;"></span>Buffer 2.000 m (&ge; 3.000 hab)</div>
</div>
"""

deck = pdk.Deck(
    layers=[layer_buffer, layer_boundary],
    initial_view_state=view,
    tooltip=tooltip,
    map_style="https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json",
)

output_html = MAP_DIR / "mapa_buffer_poblacion_huesca.html"
html = deck.to_html(as_string=True)
html = html.replace("</body>", LEGEND_HTML + "\n</body>")
with open(output_html, "w", encoding="utf-8") as f:
    f.write(html)

print(f"Abre: {output_html}")

Polígonos de buffer a dibujar: 374
Abre: ..\..\data\map\06_clasificacion_suelo\mapa_buffer_poblacion_huesca.html
